# Fashion-MNIST

**Objetivo:** usar la arquitectura ganadora del notebook de MNIST (1 capa oculta densa) para reconocer ropa (Fashion MNIST) y comparar resultados.

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 1. Carga y exploración: ¿qué datos tenemos?

Fashion MNIST trae 70.000 imágenes de 28x28 píxeles en escala de grises (60.000 para entrenar, 10.000 para test), igual formato que MNIST pero con ropa en vez de dígitos. Cada imagen tiene una etiqueta del 0 al 9 según la prenda.

In [ ]:
# Nombres de las 10 clases de Fashion MNIST (en español)
nombres_clases = ["remera", "pantalón", "suéter", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "bota"]

# Descargamos Fashion MNIST desde Keras (viene incluido)
(X_train_img, y_train), (X_test_img, y_test) = keras.datasets.fashion_mnist.load_data()

# Mostrar cantidad de imágenes y sus dimensiones
print("Train:", X_train_img.shape)
print("Test:", X_test_img.shape)

# Mostramos 10 ejemplos para ver las prendas y sus etiquetas
n_mostrar = 10
plt.figure(figsize=(10, 2))
for i in range(n_mostrar):
    plt.subplot(1, n_mostrar, i + 1)
    plt.imshow(X_train_img[i], cmap="gray")
    plt.title(nombres_clases[y_train[i]])
    plt.axis("off")
plt.show()

## 2. Preprocesamiento: normalizar y aplanar

- **Normalizar:** los píxeles van de 0 a 255. Los dividimos por 255 para dejarlos entre 0 y 1, así la red aprende más rápido (igual que en MNIST).
- **Aplanar:** la red densa no entiende imágenes 2D, así que cada foto de 28x28 la convertimos en un vector de 784 números.

In [ ]:
# Normalizamos a 0-1 y aplanamos de 28x28 a vector de 784
INPUT_DIM = 28 * 28
X_train = (X_train_img.astype("float32") / 255.0).reshape(-1, INPUT_DIM)
X_test = (X_test_img.astype("float32") / 255.0).reshape(-1, INPUT_DIM)

print("Ejemplo train aplanado:", X_train.shape)  # (60000, 784)
print("Valor mínimo y máximo:", X_train.min(), X_train.max())

## 3. Arquitectura: (ganadora de MNIST)


Repetimos el proceso con Fashion MNIST **usando la arquitectura ganadora del experimento anterior**

- **Entrada:** 784 neuronas (una por píxel).
- **Capa oculta:** 128 neuronas con activación ReLU o Sigmoid (eso lo comparamos después).
- **Salida:** 10 neuronas con `softmax` (una probabilidad por cada prenda 0-9).

La mantenemos igual a propósito para que la comparación MNIST vs Fashion sea justa: si cambia el accuracy, es por el dataset (ropa es más difícil que dígitos), no por la red.

In [ ]:
def crear_modelo(activacion, optimizador):
    lr = 0.001
    if optimizador == "adam":
        opt = keras.optimizers.Adam(learning_rate=lr)
    else:
        opt = keras.optimizers.SGD(learning_rate=lr)

    modelo = keras.Sequential([
        keras.layers.Input(shape=(784,),name="entrada"),
        keras.layers.Dense(128, activation=activacion, name="oculta"),
        keras.layers.Dense(10, activation="softmax", name="salida")
    ])

    modelo.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

# Mostramos el resumen ReLU + Adam como ejemplo (ganadora en MNIST)
modelo_demo = crear_modelo(activacion="relu", optimizador="adam")
modelo_demo.summary()

## 4. Experimentos: activaciones vs optimizadores

**Mismos hiperparámetros base para todo (igual que en MNIST):** 5 epochs, batch 128, lr 0.001, 10% de train como validación.

- Experimento 1 (activación): `ReLU + Adam` vs `Sigmoid + Adam`.
- Experimento 2 (optimizador): el ganador anterior con `Adam` vs con `SGD`.

In [ ]:
# Hiperparámetros iguales para los 3 entrenamientos para una comparación justa
epochs = 5
batch_size = 128

modelo_relu_adam = crear_modelo(activacion="relu", optimizador="adam")
hist_relu_adam = modelo_relu_adam.fit(
    X_train, y_train,
    epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose="auto"
)

modelo_sigmoid_adam = crear_modelo(activacion="sigmoid", optimizador="adam")
hist_sigmoid_adam = modelo_sigmoid_adam.fit(
    X_train, y_train,
    epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose="auto"
)

modelo_relu_sgd = crear_modelo(activacion="relu", optimizador="sgd")
hist_relu_sgd = modelo_relu_sgd.fit(
    X_train, y_train,
    epochs=epochs, batch_size=batch_size, validation_split=0.1, verbose="auto"
)

In [ ]:
# Graficamos quién baja el loss más rápido (train y validación)
plt.figure(figsize=(12, 4))

# Curva de pérdida
plt.subplot(1, 2, 1)
plt.plot(hist_relu_adam.history["loss"], label="ReLU+Adam train")
plt.plot(hist_relu_adam.history["val_loss"], linestyle="--", label="ReLU+Adam val")
plt.plot(hist_sigmoid_adam.history["loss"], label="Sigmoid+Adam train")
plt.plot(hist_sigmoid_adam.history["val_loss"], linestyle="--", label="Sigmoid+Adam val")
plt.plot(hist_relu_sgd.history["loss"], label="ReLU+SGD train")
plt.plot(hist_relu_sgd.history["val_loss"], linestyle="--", label="ReLU+SGD val")
plt.title("Loss por epoch (más bajo = mejor)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Curva de accuracy
plt.subplot(1, 2, 2)
plt.plot(hist_relu_adam.history["val_accuracy"], label="ReLU+Adam val")
plt.plot(hist_sigmoid_adam.history["val_accuracy"], label="Sigmoid+Adam val")
plt.plot(hist_relu_sgd.history["val_accuracy"], label="ReLU+SGD val")
plt.title("Accuracy de validación por epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Resumen numérico del último epoch para comparar fácil
print("Val accuracy final ReLU+Adam:", hist_relu_adam.history["val_accuracy"][-1])
print("Val accuracy final Sigmoid+Adam:", hist_sigmoid_adam.history["val_accuracy"][-1])
print("Val accuracy final ReLU+SGD:", hist_relu_sgd.history["val_accuracy"][-1])

## 5. Evaluación: pruebas de campo

Elegimos como ganador a **ReLU + Adam** (en MNIST aprendió más rápido y llegó más alto; verificamos si se repite en ropa). Lo evaluamos en test, que la red nunca vio.

In [ ]:
# El modelo ganador es ReLU + Adam
modelo_ganador = modelo_relu_adam

# Accuracy final en test (datos que la red nunca vio)
test_loss, test_acc = modelo_ganador.evaluate(X_test, y_test, verbose="auto")
print("Accuracy en test:", test_acc)

# Predecimos todas las prendas del test
y_proba = modelo_ganador.predict(X_test, verbose="auto")
y_pred = np.argmax(y_proba, axis=1)  # nos quedamos con la prenda más probable

# Matriz de confusión: filas=real, columnas=predicción. La diagonal es lo correcto.
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
plt.matshow(cm, cmap="Blues")
plt.title("Matriz de confusión (test)")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.colorbar()
plt.xticks(range(10), nombres_clases, rotation=45)
plt.yticks(range(10), nombres_clases)
plt.show()

In [ ]:
# Buscamos un acierto y un error para mostrarlos
aciertos = np.where(y_pred == y_test)[0]  # índices donde acertó
errores = np.where(y_pred != y_test)[0]  # índices donde falló
idx_acierto = aciertos[0]
idx_error = errores[0]

plt.figure(figsize=(8, 3))

# Ejemplo bien clasificado
plt.subplot(1, 2, 1)
plt.imshow(X_test_img[idx_acierto], cmap="gray")
plt.title(f"Acierto: real={nombres_clases[y_test[idx_acierto]]}, pred={nombres_clases[y_pred[idx_acierto]]}")
plt.axis("off")

# Ejemplo mal clasificado
plt.subplot(1, 2, 2)
plt.imshow(X_test_img[idx_error], cmap="gray")
plt.title(f"Error: real={nombres_clases[y_test[idx_error]]}, pred={nombres_clases[y_pred[idx_error]]}")
plt.axis("off")
plt.show()

print("Índice del error mostrado:", idx_error)